In [ ]:
"""
Semantic Chunking RAG Pipeline -- Production-Grade
===================================================
Architecture   : SemanticChunker (langchain_experimental) with ALL FOUR
                 breakpoint threshold strategies, compared side by side
LLM            : Azure OpenAI  (AzureChatOpenAI -- GPT-4o / GPT-4-turbo)
Embeddings     : HuggingFace sentence-transformers/all-MiniLM-L6-v2  (local)
Vector Store   : ChromaDB  (one persistent collection per strategy)
Data Source    : Three publicly available research PDFs from arXiv (open-access)

Reference PDFs: 
    1. "Attention Is All You Need" -- Vaswani et al., 2017
       https://arxiv.org/pdf/1706.03762.pdf

    2. "BERT: Pre-training of Deep Bidirectional Transformers" -- Devlin et al., 2018
       https://arxiv.org/pdf/1810.04805.pdf

    3. "RAG for Knowledge-Intensive NLP Tasks" -- Lewis et al., 2020
       https://arxiv.org/pdf/2005.11401.pdf

=============================================================================
Core Concept -- WHAT IS SEMANTIC CHUNKING?
=============================================================================
Every text splitting strategy before SemanticChunker splits on STRUCTURAL
signals: fixed character count, token count, paragraph breaks, heading
markers. These signals are completely agnostic to the actual MEANING of the
text. A 512-character boundary might land in the middle of an argument or
cut across two semantically unrelated topics that happen to be adjacent.

SemanticChunker splits on SEMANTIC signals. It measures the embedding-space
distance between adjacent sentence groups and places a chunk boundary
wherever the semantic shift is largest. The result is chunks that each
contain one coherent semantic unit -- a single topic, argument, or concept.
  
The algorithm (from Greg Kamradt's original notebook, adopted by LangChain):
    Step 1  -- Split the raw text into individual sentences using a regex
               that splits on sentence-ending punctuation (. ! ?).
    Step 2  -- Embed each sentence using the provided embedding model.
    Step 3  -- Create "sentence windows": for each sentence i, form a group
               consisting of sentence[i-buffer_size .. i+buffer_size].
               Default buffer_size=1 means each group is 3 sentences.
               This smooths over single noisy sentence embeddings.
    Step 4  -- Compute the cosine DISTANCE (1 - cosine_similarity) between
               adjacent sentence groups: distance[i] = dist(group_i, group_i+1).
               High distance means a topic shift; low distance means continuity.
    Step 5  -- Apply a BREAKPOINT THRESHOLD to the distance array. Positions
               where distance exceeds the threshold become chunk boundaries.
    Step 6  -- Merge all sentences between boundaries into one chunk.

Four breakpoint threshold strategies:
    PERCENTILE       (default)   threshold = Nth percentile of all distances
    STANDARD_DEVIATION           threshold = mean + N * std(distances)
    INTERQUARTILE                threshold = Q3 + N * IQR(distances)
    GRADIENT                     threshold applied to rate-of-change of distances
                                 (second derivative) -- catches sharp topic transitions

=============================================================================
Why Semantic Chunking Outperforms Fixed-Size Chunking for RAG
=============================================================================
Fixed-size chunks guarantee uniform token usage but ignore semantics. In
a research paper, section 3.1 "Model Architecture" might be 800 characters
and section 3.2 "Training Objective" 600 characters. A 512-char fixed
chunker will fuse both into one chunk AND split each in the middle. The
resulting chunk embeds poorly because its embedding vector is a confused
blend of two unrelated topics. Retrieval precision drops because queries
about architecture also surface training-related text.

SemanticChunker preserves topic coherence. The architecture chunk and the
training chunk become independent, topically pure units with clean embedding
vectors that are close to queries about their respective topics.

The trade-off: SemanticChunker requires one embedding call per sentence at
ingestion time (O(N_sentences) embeddings vs. zero for character splitters).
For a 50-page paper with ~3000 sentences, this is 3000 embedding calls.
With all-MiniLM-L6-v2 on CPU at ~14K sentences/sec, this is under 1 second
of actual compute -- entirely acceptable for any production ingestion pipeline.

"""


'\nSemantic Chunking RAG Pipeline -- Production-Grade\n===================================================\nArchitecture   : SemanticChunker (langchain_experimental) with ALL FOUR\n                 breakpoint threshold strategies, compared side by side\nLLM            : Azure OpenAI  (AzureChatOpenAI -- GPT-4o / GPT-4-turbo)\nEmbeddings     : HuggingFace sentence-transformers/all-MiniLM-L6-v2  (local)\nVector Store   : ChromaDB  (one persistent collection per strategy)\nData Source    : Three publicly available research PDFs from arXiv (open-access)\n\nReference PDFs: \n    1. "Attention Is All You Need" -- Vaswani et al., 2017\n       https://arxiv.org/pdf/1706.03762.pdf\n\n    2. "BERT: Pre-training of Deep Bidirectional Transformers" -- Devlin et al., 2018\n       https://arxiv.org/pdf/1810.04805.pdf\n\n    3. "RAG for Knowledge-Intensive NLP Tasks" -- Lewis et al., 2020\n       https://arxiv.org/pdf/2005.11401.pdf\n\n=================================================================

In [2]:
%pip install langchain_experimental langchain_text_splitters

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import time
import logging
import urllib.request
from enum import Enum
from pathlib import Path
from typing import List, Dict, Tuple, Optional

# ---------------------------------------------------------------------------
# Third-party imports
# pip install langchain langchain-community langchain-openai langchain-experimental
#             chromadb sentence-transformers pypdf
# ---------------------------------------------------------------------------
from langchain_core.documents import Document
from langchain_experimental.text_splitter import SemanticChunker
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


C:\Users\dhanu\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0407 21:05:33.151000 30644 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [4]:
# ===========================================================================
# LOGGING
# ===========================================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("semantic_chunking_rag")


In [6]:

# ===========================================================================
# SECTION 1 -- CONFIGURATION
# ===========================================================================

class BreakpointStrategy(str, Enum):
    """
    Enum for the four SemanticChunker breakpoint threshold strategies.

    PERCENTILE:        Places a boundary wherever distance > Nth percentile
                       of all pairwise distances. N defaults to 95 in LangChain.
                       Most conservative; produces fewer, larger chunks.
                       Best for: corpora where most content is topically uniform.

    STANDARD_DEVIATION: Threshold = mean(distances) + N * std(distances).
                        N defaults to 3. Sensitive to global variance in the text.
                        Best for: corpora with both dense technical and sparse
                        transitional sections (research papers, textbooks).

    INTERQUARTILE:     Threshold = Q3 + N * IQR where IQR = Q3 - Q1.
                       N defaults to 1.5 (analogous to Tukey's fence for outliers).
                       Robust to outlier distances; produces more uniform chunks.
                       Best for: noisy text with irregular sentence length.

    GRADIENT:          Threshold applied to the GRADIENT (first derivative) of
                       the cosine distance array rather than the raw distances.
                       Catches sharp semantic transitions rather than sustained
                       topic shifts. Best for: structured documents where section
                       transitions are abrupt (papers, legal docs, technical manuals).
    """
    PERCENTILE        = "percentile"
    STANDARD_DEVIATION = "standard_deviation"
    INTERQUARTILE     = "interquartile"
    GRADIENT          = "gradient"


In [7]:

class SemanticChunkingConfig:
    """
    Centralized configuration for the Semantic Chunking RAG pipeline.

    BUFFER_SIZE controls the sentence window size for smoothing.
    buffer_size=1 (default) means each window is 3 sentences (i-1, i, i+1).
    buffer_size=2 would yield 5-sentence windows.
    Larger buffer_size smooths over short noisy sentences but may blur sharp
    topic transitions. For dense technical text, buffer_size=1 is optimal.

    MIN_CHUNK_SIZE is a guard against degenerate chunks. Without it, a single
    isolated sentence could become its own chunk -- a 20-char fragment that
    embeds poorly and retrieves irrelevant context. Setting 150 chars as the
    floor merges short fragments with the preceding chunk automatically.

    BREAKPOINT_THRESHOLD_AMOUNTS maps each strategy to its threshold value.
    These are tunable per-corpus:
        percentile=95       --> only top 5% of distances trigger a boundary
        standard_deviation=3 --> 3 standard deviations above mean
        interquartile=1.5   --> Tukey's IQR fence (conservative)
        gradient=95         --> top 5% of gradient magnitudes trigger boundary
    """

    # --- Dataset -----------------------------------------------------------
    PDF_SOURCES: List[Tuple[str, str]] = [
        ("attention_is_all_you_need",    "https://arxiv.org/pdf/1706.03762.pdf"),
        ("bert_pretraining_transformers", "https://arxiv.org/pdf/1810.04805.pdf"),
        ("rag_knowledge_intensive_nlp",  "https://arxiv.org/pdf/2005.11401.pdf"),
    ]
    PDF_DOWNLOAD_DIR: str = "./pdfs"

    # --- SemanticChunker parameters (applied to ALL strategies) -----------
    BUFFER_SIZE: int              = 1       # sentence window size for smoothing
    MIN_CHUNK_SIZE: Optional[int] = 150     # minimum characters per chunk

    # Threshold amounts per strategy -- tune these for your corpus
    BREAKPOINT_THRESHOLD_AMOUNTS: Dict[str, float] = {
        BreakpointStrategy.PERCENTILE:         95.0,
        BreakpointStrategy.STANDARD_DEVIATION:  3.0,
        BreakpointStrategy.INTERQUARTILE:       1.5,
        BreakpointStrategy.GRADIENT:           95.0,
    }

    # --- Embeddings (shared by SemanticChunker AND retrieval) -------------
    # CRITICAL: Use the SAME embedding model for chunking and retrieval.
    # SemanticChunker uses it to compute sentence distances at ingestion.
    # ChromaDB uses it to embed query vectors at retrieval.
    # Mixing models is a silent correctness bug that degrades retrieval quality.
    EMBEDDING_MODEL: str  = "sentence-transformers/all-MiniLM-L6-v2"
    EMBEDDING_DEVICE: str = "cpu"           # "cuda" for GPU acceleration

    # --- ChromaDB (one collection per breakpoint strategy) ----------------
    CHROMA_PERSIST_DIR: str = "./chroma_semantic_db"
    CHROMA_COLLECTIONS: Dict[str, str] = {
        BreakpointStrategy.PERCENTILE:         "semantic_percentile",
        BreakpointStrategy.STANDARD_DEVIATION: "semantic_std_dev",
        BreakpointStrategy.INTERQUARTILE:      "semantic_iqr",
        BreakpointStrategy.GRADIENT:           "semantic_gradient",
    }

    # --- Azure OpenAI LLM -------------------------------------------------
    AZURE_OPENAI_API_KEY:    str  = os.environ["AZURE_OPENAI_API_KEY"]
    AZURE_OPENAI_ENDPOINT:   str  = os.environ["AZURE_OPENAI_ENDPOINT"]
    AZURE_OPENAI_DEPLOYMENT: str  = os.environ["AZURE_OPENAI_DEPLOYMENT"]
    AZURE_OPENAI_API_VERSION: str = os.environ["AZURE_OPENAI_API_VERSION"]
    LLM_TEMPERATURE: float        = 0.0
    LLM_MAX_TOKENS: int           = 1024

    # --- Retrieval --------------------------------------------------------
    RETRIEVER_K: int = 4    # number of chunks returned per query

    # --- Prompt template --------------------------------------------------
    RAG_PROMPT_TEMPLATE: str = """You are a precise technical research assistant.
Answer the question using ONLY the information in the context below.
If the answer cannot be found in the context, respond with:
"The provided documents do not contain enough information to answer this question."

Context (semantically chunked research paper sections):
{context}

Question: {question}

Provide a clear, technically accurate, well-structured answer:"""



In [8]:

# ===========================================================================
# SECTION 2 -- PDF DOWNLOADER
# ===========================================================================

def download_pdfs(config: SemanticChunkingConfig) -> List[Path]:
    """
    Download research PDFs from arXiv to the local filesystem.

    Skips files already cached, validates file size after download,
    and inserts a 1-second polite delay between requests to avoid
    rate-limiting from the arXiv CDN.

    Args:
        config: SemanticChunkingConfig with PDF_SOURCES and PDF_DOWNLOAD_DIR.

    Returns:
        List of Path objects pointing to successfully downloaded PDF files.

    Raises:
        RuntimeError: If a download fails or produces a suspiciously small file.
    """
    download_dir = Path(config.PDF_DOWNLOAD_DIR)
    download_dir.mkdir(parents=True, exist_ok=True)

    paths: List[Path] = []

    for name, url in config.PDF_SOURCES:
        local_path = download_dir / f"{name}.pdf"

        if local_path.exists() and local_path.stat().st_size > 10_000:
            logger.info(
                "Cached: %s  (%.1f KB)", local_path.name,
                local_path.stat().st_size / 1024,
            )
            paths.append(local_path)
            continue

        logger.info("Downloading: %s", url)
        try:
            req = urllib.request.Request(
                url,
                headers={"User-Agent": "Mozilla/5.0 (RAG-Research-Pipeline/1.0)"},
            )
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = resp.read()

            if len(data) < 10_000:
                raise RuntimeError(
                    f"File too small ({len(data)} bytes). "
                    f"Download may have failed: {url}"
                )

            local_path.write_bytes(data)
            logger.info(
                "Saved: %s  (%.1f KB)", local_path.name, len(data) / 1024
            )
            paths.append(local_path)
            time.sleep(1.0)

        except Exception as exc:
            raise RuntimeError(
                f"Cannot download '{url}'. "
                f"Manually download and place at '{local_path}'."
            ) from exc

    return paths



In [9]:

# ===========================================================================
# SECTION 3 -- PDF LOADING
# ===========================================================================

def load_pdf_documents(pdf_paths: List[Path]) -> List[Document]:
    """
    Load PDF files using PyPDFLoader and annotate each page with metadata.

    PyPDFLoader loads one Document per page. Each Document receives:
        - source     : PDF filename for provenance tracking.
        - paper_name : Human-readable title derived from filename.
        - page       : 0-indexed page number (added by PyPDFLoader).

    These metadata fields survive through SemanticChunker.split_documents()
    because SemanticChunker inherits metadata from the input Documents to
    each output chunk.

    IMPORTANT: SemanticChunker processes each input Document independently.
    It does NOT split across Document boundaries. This means page boundaries
    in PyPDFLoader output become hard boundaries for semantic chunking.
    If you want cross-page semantic analysis, concatenate adjacent page
    texts into a single Document before passing to SemanticChunker.
    For most RAG use cases, page-level independence is acceptable.

    Args:
        pdf_paths: List of Path objects pointing to downloaded PDFs.

    Returns:
        Flat list of Documents, one per page, across all PDFs.
    """
    all_docs: List[Document] = []

    for pdf_path in pdf_paths:
        paper_name = pdf_path.stem.replace("_", " ").title()
        logger.info("Loading: %s", pdf_path.name)

        try:
            loader = PyPDFLoader(str(pdf_path))
            pages  = loader.load()
        except Exception as exc:
            raise RuntimeError(f"Failed to load '{pdf_path}': {exc}") from exc

        for page_doc in pages:
            page_doc.metadata["source"]     = pdf_path.name
            page_doc.metadata["paper_name"] = paper_name
            # PyPDFLoader already sets metadata['page'] (0-indexed integer)

        logger.info("  Loaded %d pages from %s", len(pages), pdf_path.name)
        all_docs.extend(pages)

    logger.info("Total pages across all PDFs: %d", len(all_docs))
    return all_docs


In [10]:

# ===========================================================================
# SECTION 4 -- EMBEDDING MODEL
# ===========================================================================

def build_embeddings(config: SemanticChunkingConfig) -> HuggingFaceEmbeddings:
    """
    Initialize the HuggingFace sentence-transformers embedding model.

    This SAME model instance is passed to BOTH SemanticChunker (for computing
    pairwise sentence distances during chunking) AND to ChromaDB (for embedding
    query vectors and document chunks at index/retrieval time).

    Sharing the same model instance ensures consistency: the cosine distances
    that SemanticChunker uses to determine chunk boundaries are computed in
    the same 384-dimensional embedding space where ChromaDB performs retrieval.

    Passing different model objects -- even with the same model_name --
    is safe because HuggingFace caches the model weights in memory after
    the first load, so no duplicate download occurs. But sharing one instance
    avoids any risk of version skew between the chunking and retrieval steps.

    Args:
        config: SemanticChunkingConfig with EMBEDDING_MODEL and EMBEDDING_DEVICE.

    Returns:
        Initialized HuggingFaceEmbeddings instance.
    """
    logger.info(
        "Embedding model: %s  (device=%s)",
        config.EMBEDDING_MODEL, config.EMBEDDING_DEVICE,
    )
    return HuggingFaceEmbeddings(
        model_name=config.EMBEDDING_MODEL,
        model_kwargs={"device": config.EMBEDDING_DEVICE},
        encode_kwargs={"normalize_embeddings": True},  # unit norm for cosine sim
    )

In [11]:


# ===========================================================================
# SECTION 5 -- SEMANTIC CHUNKER FACTORY
# Builds one SemanticChunker per breakpoint strategy and returns the
# resulting chunks along with per-strategy statistics.
# ===========================================================================

def build_semantic_chunks(
    documents: List[Document],
    embeddings: HuggingFaceEmbeddings,
    strategy: BreakpointStrategy,
    config: SemanticChunkingConfig,
) -> Tuple[List[Document], Dict]:
    """
    Apply SemanticChunker to the raw PDF pages using the given strategy.

    SemanticChunker processes each Document independently (it does not
    merge content across Document boundaries). For each Document it:
        1. Splits text into sentences via sentence_split_regex.
        2. Embeds each sentence using the provided embeddings model.
        3. Computes cosine distances between adjacent sentence-window groups.
        4. Applies the breakpoint threshold to identify chunk boundaries.
        5. Merges sentences between boundaries into output Document chunks.
        6. Enforces min_chunk_size by merging sub-minimum chunks into their
           predecessor.

    The strategy parameter controls ONLY the threshold calculation in step 4.
    All other steps are identical across strategies.

    Performance note: SemanticChunker makes one embedding call per sentence.
    For 3 arXiv papers totaling ~300 pages and ~9000 sentences at buffer_size=1
    (3-sentence windows), this is approximately 9000 embedding calls. With
    all-MiniLM-L6-v2 at ~14K sentences/sec on CPU, expect ~1-2 seconds total
    compute per strategy. The model is already loaded into memory so there
    is no repeated download cost across strategies.

    Args:
        documents  : Raw page Documents from PyPDFLoader.
        embeddings : Initialized HuggingFaceEmbeddings (same instance used
                     for vector store retrieval).
        strategy   : BreakpointStrategy enum value.
        config     : SemanticChunkingConfig with threshold amounts and buffer_size.

    Returns:
        Tuple of:
            - List of semantically chunked Documents with metadata.
            - Stats dict with chunk count, avg/min/max chunk sizes.
    """
    threshold_amount = config.BREAKPOINT_THRESHOLD_AMOUNTS[strategy]

    logger.info(
        "Chunking with strategy='%s'  threshold_amount=%.1f  buffer_size=%d  min_chunk_size=%s",
        strategy.value, threshold_amount, config.BUFFER_SIZE,
        str(config.MIN_CHUNK_SIZE),
    )

    chunker = SemanticChunker(
        embeddings=embeddings,
        buffer_size=config.BUFFER_SIZE,
        breakpoint_threshold_type=strategy.value,
        breakpoint_threshold_amount=threshold_amount,
        min_chunk_size=config.MIN_CHUNK_SIZE,
        add_start_index=True,   # adds metadata['start_index'] for traceability
    )

    start_time = time.perf_counter()
    chunks = chunker.split_documents(documents)
    elapsed = time.perf_counter() - start_time

    # Compute statistics for comparison
    chunk_lengths = [len(c.page_content) for c in chunks]
    stats = {
        "strategy":      strategy.value,
        "n_chunks":      len(chunks),
        "avg_chars":     int(sum(chunk_lengths) / max(len(chunk_lengths), 1)),
        "min_chars":     min(chunk_lengths) if chunk_lengths else 0,
        "max_chars":     max(chunk_lengths) if chunk_lengths else 0,
        "elapsed_sec":   round(elapsed, 2),
        "input_pages":   len(documents),
    }

    logger.info(
        "  Strategy '%s': %d chunks  avg=%d chars  min=%d  max=%d  (%.2fs)",
        strategy.value, stats["n_chunks"], stats["avg_chars"],
        stats["min_chars"], stats["max_chars"], stats["elapsed_sec"],
    )

    return chunks, stats


In [12]:

# ===========================================================================
# SECTION 6 -- CHROMADB VECTOR STORE BUILDER
# One ChromaDB collection per strategy so all four can be queried and
# compared independently for the same question.
# ===========================================================================

def build_vectorstore(
    chunks: List[Document],
    embeddings: HuggingFaceEmbeddings,
    strategy: BreakpointStrategy,
    config: SemanticChunkingConfig,
    force_rebuild: bool = False,
) -> Chroma:
    """
    Build or load a ChromaDB collection for the given strategy's chunks.

    ChromaDB persists to disk at CHROMA_PERSIST_DIR. If the collection for
    this strategy already exists (detected by checking the count of stored
    vectors), we load it from disk rather than re-embedding, saving time
    and preserving consistency between runs.

    Each strategy gets its own ChromaDB collection name so the four indexes
    are fully independent and can be queried and compared in isolation.

    The chunks passed here are the semantically split Documents. Their
    page_content is embedded by HuggingFace and stored in ChromaDB. The
    metadata (paper_name, source, page, start_index) is stored alongside
    each vector and returned with search results, enabling provenance tracking.

    Args:
        chunks        : Semantically chunked Documents for this strategy.
        embeddings    : HuggingFace embedding model.
        strategy      : BreakpointStrategy that produced these chunks.
        config        : SemanticChunkingConfig with collection names.
        force_rebuild : If True, drop existing collection and rebuild.

    Returns:
        Populated Chroma vector store for this strategy.
    """
    collection_name = config.CHROMA_COLLECTIONS[strategy]

    # Attempt to load an existing collection first
    if not force_rebuild:
        try:
            existing = Chroma(
                collection_name=collection_name,
                embedding_function=embeddings,
                persist_directory=config.CHROMA_PERSIST_DIR,
            )
            count = existing._collection.count()
            if count > 0:
                logger.info(
                    "Loaded existing ChromaDB collection '%s'  (%d vectors)",
                    collection_name, count,
                )
                return existing
        except Exception:
            pass   # collection does not exist yet -- fall through to create

    logger.info(
        "Building ChromaDB collection '%s' from %d semantic chunks ...",
        collection_name, len(chunks),
    )

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=collection_name,
        persist_directory=config.CHROMA_PERSIST_DIR,
    )

    logger.info(
        "Collection '%s' built.  Vectors stored: %d",
        collection_name, vectorstore._collection.count(),
    )
    return vectorstore


In [13]:

# ===========================================================================
# SECTION 7 -- AZURE OPENAI LLM
# ===========================================================================

def build_azure_llm(config: SemanticChunkingConfig) -> AzureChatOpenAI:
    """
    Initialize and validate the Azure OpenAI chat model.

    Performs credential validation upfront so the pipeline fails immediately
    with a clear error message rather than failing mid-execution after
    minutes of embedding computation.

    Required environment variables:
        AZURE_OPENAI_API_KEY      : Azure resource key
        AZURE_OPENAI_ENDPOINT     : https://<resource>.openai.azure.com/
        AZURE_OPENAI_DEPLOYMENT   : Your deployment name in Azure AI Studio
        AZURE_OPENAI_API_VERSION  : e.g. 2024-02-15-preview

    Args:
        config: SemanticChunkingConfig with Azure credentials.

    Returns:
        Initialized AzureChatOpenAI instance.

    Raises:
        EnvironmentError: If required credentials are absent.
    """
    missing = [
        name for name, val in [
            ("AZURE_OPENAI_API_KEY",  config.AZURE_OPENAI_API_KEY),
            ("AZURE_OPENAI_ENDPOINT", config.AZURE_OPENAI_ENDPOINT),
        ] if not val
    ]
    if missing:
        raise EnvironmentError(
            f"Missing required environment variables: {missing}\n\n"
            "Set them before running:\n"
            "  export AZURE_OPENAI_API_KEY='your-key'\n"
            "  export AZURE_OPENAI_ENDPOINT='https://your-resource.openai.azure.com/'\n"
            "  export AZURE_OPENAI_DEPLOYMENT='gpt-4o'\n"
            "  export AZURE_OPENAI_API_VERSION='2024-02-15-preview'"
        )

    logger.info(
        "Azure OpenAI LLM: deployment='%s'  api_version='%s'",
        config.AZURE_OPENAI_DEPLOYMENT, config.AZURE_OPENAI_API_VERSION,
    )

    return AzureChatOpenAI(
        azure_deployment=config.AZURE_OPENAI_DEPLOYMENT,
        azure_endpoint=config.AZURE_OPENAI_ENDPOINT,
        api_key=config.AZURE_OPENAI_API_KEY,
        api_version=config.AZURE_OPENAI_API_VERSION,
        temperature=config.LLM_TEMPERATURE,
        max_tokens=config.LLM_MAX_TOKENS,
    )

In [14]:

# ===========================================================================
# SECTION 8 -- RAG CHAIN FACTORY
# A single factory that builds an identical LCEL RAG chain for any
# ChromaDB vector store, making strategy comparison completely uniform.
# ===========================================================================

def build_rag_chain(
    vectorstore: Chroma,
    llm: AzureChatOpenAI,
    config: SemanticChunkingConfig,
    strategy_label: str = "",
):
    """
    Build a complete LCEL RAG chain wrapping the given ChromaDB vector store.

    The retriever uses similarity search (cosine) to find the top-k semantically
    chunked Documents closest to the query embedding. Because semantic chunks
    are topically coherent, each retrieved chunk is relevant to a distinct
    facet of the query rather than the same topic reformulated five times
    (a common failure mode with fixed-size chunking where adjacent overlapping
    chunks rank first and dominate the context window).

    Context formatting annotates each chunk with its source paper, page number,
    and character length, giving the LLM explicit provenance information that
    it can reference in answers ("According to the Transformer paper, page 4...").

    Args:
        vectorstore    : Populated Chroma vector store for one strategy.
        llm            : AzureChatOpenAI instance.
        config         : SemanticChunkingConfig with RETRIEVER_K and prompt.
        strategy_label : Human-readable label for logging and display.

    Returns:
        Tuple of (LCEL rag_chain, retriever).
    """
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": config.RETRIEVER_K},
    )

    prompt = ChatPromptTemplate.from_template(config.RAG_PROMPT_TEMPLATE)

    def format_docs(docs: List[Document]) -> str:
        """
        Format retrieved semantic chunks into an annotated context block.

        Each chunk is labeled with its paper, page, character length, and
        start_index (character offset in the original page). The length label
        helps the LLM and the developer understand the density of each chunk.
        Semantic chunks vary in length (10 chars to 2000+ chars) unlike
        fixed-size chunks, so this label is informative.
        """
        parts = []
        for i, doc in enumerate(docs, start=1):
            paper  = doc.metadata.get("paper_name", doc.metadata.get("source", "Unknown"))
            page   = doc.metadata.get("page", "?")
            chars  = len(doc.page_content)
            start  = doc.metadata.get("start_index", "?")
            text   = doc.page_content.strip()
            parts.append(
                f"[Chunk {i} | {paper} | Page {page} | {chars} chars | offset {start}]\n{text}"
            )
        separator = "\n\n" + ("-" * 60) + "\n\n"
        return separator.join(parts)

    rag_chain = (
        {
            "context":  retriever | format_docs,
            "question": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    logger.info("RAG chain ready: %s", strategy_label or "unnamed strategy")
    return rag_chain, retriever



In [15]:

# ===========================================================================
# SECTION 9 -- CHUNK ANALYSIS REPORTER
# Prints a detailed statistical comparison of all four strategies to help
# developers choose the right strategy for their corpus.
# ===========================================================================

def print_chunk_analysis(all_stats: List[Dict]) -> None:
    """
    Print a formatted comparison table of chunking statistics.

    This report is the primary diagnostic tool for selecting a breakpoint
    strategy. The ideal strategy for a given corpus produces:
        - n_chunks proportional to the number of distinct topics/sections
        - avg_chars large enough to give the LLM useful context (> 200 chars)
        - min_chars above the min_chunk_size floor (validates the guard works)
        - max_chars not excessively large (< 3000 chars avoids token overflow)

    Args:
        all_stats: List of stats dicts returned by build_semantic_chunks().
    """
    print("\n" + "=" * 80)
    print("SEMANTIC CHUNKING STRATEGY COMPARISON")
    print("=" * 80)
    print(
        f"{'Strategy':<22} {'Chunks':>8} {'Avg(chars)':>12} "
        f"{'Min(chars)':>12} {'Max(chars)':>12} {'Time(s)':>8}"
    )
    print("-" * 80)
    for s in all_stats:
        print(
            f"{s['strategy']:<22} {s['n_chunks']:>8} {s['avg_chars']:>12} "
            f"{s['min_chars']:>12} {s['max_chars']:>12} {s['elapsed_sec']:>8.2f}"
        )
    print("=" * 80)

    # Recommendation logic
    best = min(all_stats, key=lambda x: abs(x["avg_chars"] - 600))
    print(
        f"\nRecommendation: '{best['strategy']}' produces chunks closest to "
        f"the ~600-char target for balanced retrieval precision and LLM context."
    )
    print(
        "Note: For your specific corpus and query distribution, evaluate all "
        "strategies against a held-out query set and measure RAGAS scores.\n"
    )



In [16]:

# ===========================================================================
# SECTION 10 -- COMPARATIVE QUERY RUNNER
# ===========================================================================

def query_semantic_rag(
    chain,
    retriever,
    question: str,
    strategy_label: str,
    show_sources: bool = True,
) -> str:
    """
    Execute a question against one semantic chunking RAG chain.

    Retrieves and displays the source chunks before generating the LLM answer.
    Note that the chunk sizes displayed here will vary significantly across
    strategies -- this is expected and demonstrates the semantic coherence
    vs. chunk count trade-off.

    Args:
        chain          : LCEL RAG chain.
        retriever      : ChromaDB-backed retriever for source display.
        question       : Natural language question string.
        strategy_label : Display label for the current strategy.
        show_sources   : If True, print retrieved chunk metadata.

    Returns:
        LLM-generated answer string.
    """
    logger.info("[%s] Query: '%s'", strategy_label, question)

    if show_sources:
        source_chunks = retriever.invoke(question)
        print("\n" + "=" * 72)
        print(f"STRATEGY : {strategy_label}")
        print(f"QUERY    : {question}")
        print("=" * 72)
        print(f"\nRETRIEVED SEMANTIC CHUNKS ({len(source_chunks)}):")
        for i, chunk in enumerate(source_chunks, start=1):
            paper  = chunk.metadata.get("paper_name", "Unknown")
            page   = chunk.metadata.get("page", "?")
            chars  = len(chunk.page_content)
            start  = chunk.metadata.get("start_index", "?")
            snippet = chunk.page_content[:250].replace("\n", " ").strip()
            print(
                f"\n  [{i}] {paper} | Page {page} | {chars} chars | offset {start}"
            )
            print(f"       {snippet}...")

    answer = chain.invoke(question)

    if show_sources:
        print(f"\nANSWER:\n{answer}")
        print("=" * 72 + "\n")

    return answer


def compare_all_strategies(
    chains_and_retrievers: Dict[str, tuple],
    question: str,
) -> Dict[str, str]:
    """
    Run the same question through all four semantic chunking strategies.

    Collects and returns answers for side-by-side comparison. In a production
    evaluation workflow, feed these results to an LLM-as-judge or RAGAS scorer
    to identify which strategy maximizes faithfulness and answer relevance for
    your specific domain and query distribution.

    Args:
        chains_and_retrievers : Dict mapping strategy label to (chain, retriever).
        question              : Query string to run across all strategies.

    Returns:
        Dict mapping strategy label to answer string.
    """
    results: Dict[str, str] = {}
    for label, (chain, retriever) in chains_and_retrievers.items():
        results[label] = query_semantic_rag(
            chain, retriever, question, label, show_sources=True
        )
    return results



In [17]:

# ===========================================================================
# SECTION 11 -- MAIN ORCHESTRATOR
# ===========================================================================

def main() -> None:
    """
    End-to-end Semantic Chunking RAG pipeline orchestrator.

    Execution order:
        1. Download three arXiv PDFs (cached after first run).
        2. Load pages using PyPDFLoader.
        3. Initialize HuggingFace embedding model (local, shared instance).
        4. Validate and initialize Azure OpenAI LLM.
        5. For each of the four breakpoint strategies:
             a. Apply SemanticChunker to all pages --> strategy-specific chunks.
             b. Build/load ChromaDB collection for those chunks.
             c. Build LCEL RAG chain.
        6. Print cross-strategy chunk statistics comparison table.
        7. Run demo queries through all four strategies for side-by-side comparison.

    Design decision -- why initialize embeddings BEFORE the LLM:
        The embedding model is needed immediately for SemanticChunker (step 5a).
        The LLM is only needed for chain execution (step 7). Initializing the
        LLM first (step 4) catches credential errors early, before any expensive
        embedding computation, but embedding model loading is done at step 3 so
        it can be shared across all four SemanticChunker instances.
    """
    config = SemanticChunkingConfig()
    logger.info("=== Semantic Chunking RAG Pipeline Starting ===")

    # Steps 1-2: Data
    pdf_paths = download_pdfs(config)
    documents = load_pdf_documents(pdf_paths)

    # Step 3: Embeddings (single shared instance -- see docstring for why)
    embeddings = build_embeddings(config)

    # Step 4: LLM (validates Azure credentials early, before chunking work)
    llm = build_azure_llm(config)

    # Step 5: Run all four strategies
    all_stats: List[Dict] = []
    chains_and_retrievers: Dict[str, tuple] = {}

    strategies = [
        BreakpointStrategy.PERCENTILE,
        BreakpointStrategy.STANDARD_DEVIATION,
        BreakpointStrategy.INTERQUARTILE,
        BreakpointStrategy.GRADIENT,
    ]

    for strategy in strategies:
        label = f"SemanticChunker ({strategy.value})"

        # 5a: Chunk with this strategy
        chunks, stats = build_semantic_chunks(documents, embeddings, strategy, config)
        all_stats.append(stats)

        # 5b: Build or load ChromaDB collection
        vectorstore = build_vectorstore(chunks, embeddings, strategy, config)

        # 5c: Build RAG chain
        chain, retriever = build_rag_chain(vectorstore, llm, config, label)
        chains_and_retrievers[label] = (chain, retriever)

    # Step 6: Print strategy comparison table
    print_chunk_analysis(all_stats)

    # Step 7: Demo queries -- each runs through all four strategies
    demo_questions = [
        # Multi-sentence technical explanation (tests chunk coherence)
        "What is the scaled dot-product attention mechanism and why is scaling used?",

        # Specific numerical fact (tests retrieval precision)
        "What embedding dimension and number of attention heads does the base Transformer model use?",

        # Conceptual cross-paper question (tests semantic coverage)
        "How does BERT's bidirectional training objective improve on the original Transformer's pre-training approach?",

        # RAG-specific technical question (tests the Lewis et al. paper)
        "What are the two RAG formulations described in the RAG paper and how do they differ at inference time?",
    ]

    logger.info(
        "Running %d queries across all %d strategies ...",
        len(demo_questions), len(strategies),
    )

    for question in demo_questions:
        compare_all_strategies(chains_and_retrievers, question)

    logger.info("=== Semantic Chunking RAG Pipeline Demo Complete ===")



In [18]:
main()

2026-04-04 19:54:16 | INFO     | semantic_chunking_rag | === Semantic Chunking RAG Pipeline Starting ===
2026-04-04 19:54:16 | INFO     | semantic_chunking_rag | Cached: attention_is_all_you_need.pdf  (2163.3 KB)
2026-04-04 19:54:16 | INFO     | semantic_chunking_rag | Cached: bert_pretraining_transformers.pdf  (757.0 KB)
2026-04-04 19:54:16 | INFO     | semantic_chunking_rag | Cached: rag_knowledge_intensive_nlp.pdf  (864.6 KB)
2026-04-04 19:54:16 | INFO     | semantic_chunking_rag | Loading: attention_is_all_you_need.pdf
2026-04-04 19:54:17 | INFO     | semantic_chunking_rag |   Loaded 15 pages from attention_is_all_you_need.pdf
2026-04-04 19:54:17 | INFO     | semantic_chunking_rag | Loading: bert_pretraining_transformers.pdf
2026-04-04 19:54:18 | INFO     | semantic_chunking_rag |   Loaded 16 pages from bert_pretraining_transformers.pdf
2026-04-04 19:54:18 | INFO     | semantic_chunking_rag | Loading: rag_knowledge_intensive_nlp.pdf
2026-04-04 19:54:18 | INFO     | semantic_chunkin

C:\Users\dhanu\AppData\Local\Temp\ipykernel_18672\1640231060.py:32: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(


2026-04-04 19:54:19 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-04-04 19:54:19 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
2026-04-04 19:54:19 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-04-04 19:54:19 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-04-04 19:54:19 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_trans

2026-04-04 19:54:20 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-04-04 19:54:20 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/sentence_bert_config.json "HTTP/1.1 200 OK"
2026-04-04 19:54:20 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-04-04 19:54:21 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-04 19:54:21 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6236.39it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-04-04 19:54:22 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-04 19:54:22 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config.json "HTTP/1.1 200 OK"
2026-04-04 19:54:22 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-04 19:54:22 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/tokenizer_config.json "HTTP/1.1 200 OK"
2026-04-04 19:54:23 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=fals